# TV1 Week 4 — Regression Master

**Project:** Retake Behavior & Recovery Analysis  
**Goal:** Baseline + 5 regression models + 5-fold group cross-validation + tuning + leakage check.

> Notebook này được viết để dễ thuyết trình: mỗi bước đều có chú thích về *làm gì*, *tại sao làm* và *cách đọc kết quả*.

## 1. Target và features

- **Target:** `score_delta = next_attempt_total - initial_failure_total`
- **Predictors:** `initial_failure_total`, `initial_failure_type`, `time_gap`, `prior_attempt_count`, `course_id`
- Không dùng `next_attempt_total`, `next_status_group`, `pass_next_attempt` vì đây là thông tin kết quả/tương lai.

`time_gap` phù hợp với mô hình ở **retake stage** (khi thời điểm retake đã biết), không nên diễn giải là predictor có sẵn ngay tại thời điểm vừa fail.

In [13]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

BASE = Path.cwd()
canonical = BASE.parent / 'model_ready_v1.csv'
fallback = BASE.parent / 'model_ready_v1_TV1_fallback.csv'
DATA = canonical if canonical.exists() else fallback
print('Dataset used:', DATA)
df = pd.read_csv(DATA)
print('Shape:', df.shape)
df.head()

Dataset used: c:\Users\PC\Documents\GitHub\ADY201m_Retake_Recovery\model_ready_v1_TV1_fallback.csv


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\PC\\Documents\\GitHub\\ADY201m_Retake_Recovery\\model_ready_v1_TV1_fallback.csv'

## 2. QA + Data Leakage Check

TV1 kiểm tra dataset trước khi model. `student_id` chỉ được dùng làm **group** để chia train/test/CV, không dùng làm predictor.

In [2]:
features = ['initial_failure_total','initial_failure_type','time_gap','prior_attempt_count','course_id']
target = 'score_delta'
forbidden = {'next_attempt_total','next_status_group','pass_next_attempt','score_delta'}

print('Missing values:')
print(df[['student_id'] + features + [target]].isna().sum())
print('\nPrior attempt count distribution:')
print(df['prior_attempt_count'].value_counts(dropna=False).sort_index())

assert not (set(features) & forbidden), 'DATA LEAKAGE: forbidden feature detected.'
print('\nLeakage feature check: PASSED')

Missing values:
student_id               0
initial_failure_total    0
initial_failure_type     0
time_gap                 0
prior_attempt_count      0
course_id                0
score_delta              0
dtype: int64

Prior attempt count distribution:
prior_attempt_count
0    10781
1        5
Name: count, dtype: int64

Leakage feature check: PASSED


## 3. Group-aware Train/Test Split

Thay vì chia ngẫu nhiên theo từng row, dùng `student_id` làm group. Như vậy cùng một sinh viên không xuất hiện ở cả train và test, giúp đánh giá nghiêm ngặt hơn khi một sinh viên có thể có nhiều course trajectories.

In [3]:
X = df[features]
y = df[target]
groups = df['student_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]

print('Train rows:', len(train_idx))
print('Test rows :', len(test_idx))
print('Train students:', groups_train.nunique())
print('Test students :', groups_test.nunique())
print('Student overlap:', len(set(groups_train) & set(groups_test)))

Train rows: 8557
Test rows : 2229
Train students: 4630
Test students : 1158
Student overlap: 0


## 4. Preprocessing chung

- Numeric: median imputation + StandardScaler.
- Categorical: most-frequent imputation + OneHotEncoder.
- Tất cả model dùng **cùng preprocessing** để so sánh công bằng.

In [4]:
numeric_features = ['initial_failure_total','time_gap','prior_attempt_count']
categorical_features = ['initial_failure_type','course_id']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

def make_pipeline(model):
    return Pipeline([('preprocessor', preprocessor), ('model', model)])

## 5. Baseline + 5 regression models

Baseline trả lời câu hỏi: **nếu chỉ đoán Score Delta trung bình thì sai số là bao nhiêu?** Các model thật cần được so với mốc này.

In [5]:
models = {
    'Dummy Baseline': make_pipeline(DummyRegressor(strategy='mean')),
    'Linear Regression': make_pipeline(LinearRegression()),
    'Ridge Regression': make_pipeline(Ridge(alpha=1.0)),
    'Lasso Regression': make_pipeline(Lasso(alpha=0.01, max_iter=10000)),
    'Random Forest': make_pipeline(RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)),
    'Gradient Boosting': make_pipeline(GradientBoostingRegressor(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42))
}
list(models.keys())

['Dummy Baseline',
 'Linear Regression',
 'Ridge Regression',
 'Lasso Regression',
 'Random Forest',
 'Gradient Boosting']

## 6. 5-fold Group Cross-Validation + Test Metrics

- CV metric: RMSE (lower is better)
- Final test metrics: MAE, RMSE, R²
- `GroupKFold` tiếp tục giữ sinh viên tách biệt giữa các folds.

In [6]:
gkf = GroupKFold(n_splits=5)
results = []
fitted_models = {}

for name, model in models.items():
    cv_rmse = -cross_val_score(
        model, X_train, y_train,
        cv=gkf, groups=groups_train,
        scoring='neg_root_mean_squared_error', n_jobs=-1
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        'Model': name,
        'CV_RMSE_Mean': cv_rmse.mean(),
        'CV_RMSE_SD': cv_rmse.std(),
        'Test_MAE': mean_absolute_error(y_test, pred),
        'Test_RMSE': mean_squared_error(y_test, pred) ** 0.5,
        'Test_R2': r2_score(y_test, pred)
    })
    fitted_models[name] = model

results_df = pd.DataFrame(results).sort_values('Test_RMSE')
results_df.round(4)

,Model,CV_RMSE_Mean,CV_RMSE_SD,Test_MAE,Test_RMSE,Test_R2
5,Gradient Boosting,2.4548,0.0409,2.0555,2.4895,0.1585
2,Ridge Regression,2.4773,0.0433,2.0791,2.4957,0.1543
1,Linear Regression,2.4772,0.0432,2.0785,2.4957,0.1543
3,Lasso Regression,2.4818,0.0467,2.0885,2.5038,0.1488
4,Random Forest,2.4910,0.0486,2.0690,2.5203,0.1375
0,Dummy Baseline,2.7031,0.0503,2.1844,2.7147,-0.0006


## 7. Ridge Tuning

TV1 tuning `alpha` bằng `GridSearchCV`. Không cần grid quá lớn; mục tiêu là chứng minh quy trình tuning đúng và reproducible.

In [7]:
ridge_pipe = make_pipeline(Ridge())
param_grid = {'model__alpha':[0.01,0.1,1,10,100]}

grid = GridSearchCV(
    ridge_pipe, param_grid,
    cv=gkf, scoring='neg_root_mean_squared_error', n_jobs=-1
)
grid.fit(X_train, y_train, groups=groups_train)

print('Best alpha:', grid.best_params_['model__alpha'])
print('Best CV RMSE:', -grid.best_score_)

ridge_pred = grid.best_estimator_.predict(X_test)
print('Test MAE :', mean_absolute_error(y_test, ridge_pred))
print('Test RMSE:', mean_squared_error(y_test, ridge_pred)**0.5)
print('Test R2  :', r2_score(y_test, ridge_pred))

Best alpha: 0.1
Best CV RMSE: 2.477229886972464
Test MAE : 2.0785728381150075
Test RMSE: 2.495678842520272
Test R2  : 0.1543061593590047


## 8. Model comparison chart

Chart này dùng cho Week 5 Review 1. Không chỉ nhìn chart; khi trình bày cần nói cả CV RMSE và test metrics.

In [8]:
plot_df = results_df.sort_values('Test_RMSE')
plt.figure(figsize=(9,5))
plt.bar(plot_df['Model'], plot_df['Test_RMSE'])
plt.title('Week 4 Model Comparison — Test RMSE')
plt.ylabel('RMSE (lower is better)')
plt.xlabel('Model')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

NameError: name 'results_df' is not defined

## 9. Interpretation rules for presentation

- `RMSE/MAE ↓` tốt hơn; `R² ↑` tốt hơn.
- Không nói feature/model **gây ra** recovery hay score improvement.
- Kết quả là predictive association trên dữ liệu quan sát.
- `prior_attempt_count` gần như luôn bằng 0 trong first-failure cohort → limitation.
- Không dùng một test split duy nhất để khẳng định model tốt; xem thêm 5-fold CV.